In [ ]:
import pandas as pd
import numpy as np
import scipy.stats as sps

In [ ]:
data = pd.read_csv('/content/golden_mine.csv')

In [ ]:
data = data[data['Profit'].abs() < 1000000].copy()

In [ ]:
cost = 15000
alfa = 0.05
power = 0.8
n_plan = 1000

In [ ]:
#анализ без таргетирования
mu_total = data['Profit'].mean()
var_total = data['Profit'].var()
n_total = len(data)

In [ ]:
print(f"Количество клиентов: {n_total}")
print(f"Прибыль от одного клиента: {mu_total}")
print(f"Количество привлекаемых клиентов: {n_plan}")
print(f"Цена привлечения одного клиента: {cost}")

Количество клиентов: 8713
Прибыль от одного клиента: 15755.095833811545
Количество привлекаемых клиентов: 1000
Цена привлечения одного клиента: 15000


In [ ]:
z_val = (mu_total - cost) / (var_total / n_total)**0.5
p_result = 1 - sps.norm.cdf(abs(z_val))

print(f"Статистика Z: {z_val:.4f} | P-значение: {p_result:.5f}")

Статистика Z: 3.5021 | P-значение: 0.00023


In [ ]:
#Вывод: p-value < z => можно привлекать клиентов без таргетирования, тк окупится

In [ ]:
#Таргетирование по полу

In [ ]:
# Фильтрация по половозрастным группам
m = data[(data['Gender'] == "male") & (data['AgeGroup'] == "18-24")]
w = data[(data['Gender'] == "female") & (data['AgeGroup'] == "18-24")]

# Сравнение доходности между полами
E_m = m['Profit'].mean()
E_w = w['Profit'].mean()
V_m = np.var(m['Profit'], ddof=1)
V_w = np.var(w['Profit'], ddof=1)
z_stat = (E_m - E_w) / (V_m / len(m) + V_w / len(w))**0.5
p_value = sps.norm.cdf(z_stat)
z_stat, round(p_value, 20)

(np.float64(-27.09280730417151), np.float64(0.0))

In [ ]:
#получается, женщины приносят больше прибыли, значит, они выгоднее

In [ ]:
z_stat_m = (E_m - cost) / ((V_m / len(m))**0.5)
p_value_m = 1 - sps.norm.cdf(abs(z_stat_m))

z_stat_w = (E_w - cost) / ((V_w / len(m))**0.5)
p_value_w = 1 - sps.norm.cdf(abs(z_stat_w))

print(f"z-статистика мужчин: {z_stat_m:.10f},\nz-статистика женщин: {z_stat_w:.10f},\np-value мужчин: {p_value_m:.10f},\np-value женщин: {p_value_w:.10f}")

z-статистика мужчин: -4.0650770660,
z-статистика женщин: 77.4343116317,
p-value мужчин: 0.0000240083,
p-value женщин: 0.0000000000


In [ ]:
#привлечение мужчин не окупится, а женщин точно окупится с тем же уровнем значимости

Таргетирование женщин по возрасту


In [ ]:
w_18_21 = w[(18 <= w['Age']) & (w['Age'] <= 21)]
w_22_24 = w[(22 <= w['Age']) & (w['Age'] <= 24)]

In [ ]:
E_18_21 = w_18_21['Profit'].mean()
E_22_24 = w_22_24['Profit'].mean()

V_18_21 = np.var(w_18_21['Profit'], ddof=1)
V_22_24 = np.var(w_22_24['Profit'], ddof=1)

z_stat = (E_18_21 - E_22_24) / ((V_18_21 / len(w_18_21) + V_22_24 / len(w_22_24))**0.5)
p_value = 1 - sps.norm.cdf(abs(z_stat))
z_stat, p_value

(np.float64(-8.481210212854764), np.float64(0.0))

In [ ]:
#Узкое таргетирование по возрасту
def get_group(gender_opt, age_range):
    return data[data['Gender'].isin(gender_opt) & data['Age'].isin(age_range)]

lst = [
    {'g': ['male'], 'a': [18, 19]}, {'g': ['male'], 'a': [20, 21]},
    {'g': ['male'], 'a': [22, 23]}, {'g': ['male'], 'a': [21, 24]},
    {'g': ['female'], 'a': [18, 19]}, {'g': ['female'], 'a': [20, 21]},
    {'g': ['female'], 'a': [21, 22]}, {'g': ['female'], 'a': [23, 24]}
]

top_profit = -1e9
best_ind = 0

for i, item in enumerate(lst):
    subset = get_group(item['g'], item['a'])
    if not subset.empty:
        m_val = subset['Profit'].mean()
        if m_val > top_profit:
            top_profit = m_val
            best_ind = i

winner = lst[best_ind]
print(f"Лидирующая группа: {winner['g']} | Возраст: {winner['a']}")
print(f"Средняя прибыль сегмента: {top_profit:.2f}")


Лидирующая группа: ['male'] | Возраст: [22, 23]
Средняя прибыль сегмента: 80288.11


In [ ]:
#как итог Золотая жила - мужчины 22-23 года

In [ ]:
#есть еще группа женщин 18-24, сравним
female = get_group(["female"], np.arange(18, 25, 1))
gold = get_group(winner['g'], winner['a'])

E1 = female['Profit'].mean()
E2 = gold['Profit'].mean()

V1 = np.var(female['Profit'], ddof=1)
V2 = np.var(gold['Profit'], ddof=1)

z_stat = (E1 - E2) / ((V1 / len(female) + V2 / len(gold))**0.5)
p_value = 1 - sps.norm.cdf(abs(z_stat))
z_stat, p_value

(np.float64(-62.8823811438732), np.float64(0.0))

In [ ]:
#в итоге всё равно Золотая жила выгоднее

MDE


In [ ]:
m18_data = data.query("Gender == 'male' and Age == 18")['Profit']
variance_m18 = m18_data.var()
count_m18 = len(m18_data)

q_alpha = sps.norm.ppf(1 - alfa)
q_beta = sps.norm.ppf(power)

mde_res = (q_alpha + q_beta) * np.sqrt(variance_m18 / count_m18)
print(f"Значение MDE для сегмента M18: {mde_res:.4f}")


Значение MDE для сегмента M18: 144.9093
